In [3]:
"""
3 Hedefli Deney — Kaggle S6E3 Churn
Amacı: 0.91378'i geçmek

Deney 1: max_depth=1 (Chris Deotte'nin önerisi — bu veri için kanıtlandı)
Deney 2: Orijinal veriyi SATIR olarak değil SÜTUN olarak eklemek (groupby mean, daha agresif)
Deney 3: Rank-based normalizasyon (sayısal feature'ları sıraya göre normalize et)
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# VERİ YÜKLEME
# ─────────────────────────────────────────────
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")

try:
    orig = pd.read_csv(
        "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    )
    orig["TotalCharges"] = pd.to_numeric(orig["TotalCharges"], errors="coerce")
    orig["Churn_bin"] = (orig["Churn"] == "Yes").astype(int)
    has_orig = True
    print("✅ Orijinal IBM verisi yüklendi.")
except:
    has_orig = False
    print("⚠️ Orijinal veri yüklenemedi.")

TARGET = "Churn"
ID_COL = "id"

# ─────────────────────────────────────────────
# ÖN İŞLEME
# ─────────────────────────────────────────────
def preprocess(df):
    df = df.copy()
    df["TotalCharges"] = pd.to_numeric(df.get("TotalCharges", np.nan), errors="coerce")
    df["TotalCharges"].fillna(
        df.get("MonthlyCharges", 0) * df.get("tenure", 0), inplace=True
    )
    return df

train = preprocess(train)
test  = preprocess(test)

# Kategorik encode
cat_cols = [c for c in train.select_dtypes("object").columns if c not in [TARGET, ID_COL]]
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]]).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# ─────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────
def add_features(df):
    df = df.copy()
    df["Total_Extra_Services"] = (
        df.get("OnlineSecurity", 0) + df.get("OnlineBackup", 0) +
        df.get("DeviceProtection", 0) + df.get("TechSupport", 0) +
        df.get("StreamingTV", 0) + df.get("StreamingMovies", 0)
    )
    if "tenure" in df.columns:
        df["Tenure_Contract_Ratio"] = df["tenure"] / (df.get("Contract", 0) + 1)
        df["Charge_Per_Month"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["TotalCharges_Diff"] = df["TotalCharges"] - (
        df.get("tenure", 0) * df.get("MonthlyCharges", 0)
    )
    df["Has_Family"] = df.get("Dependents", 0) + df.get("Partner", 0)
    return df

train = add_features(train)
test  = add_features(test)

# ── DENEY 2: Orijinal veriden agresif groupby feature'ları ──
if has_orig:
    # Tek kolon yerine çoklu kombinasyonlar
    merge_cols = ["MonthlyCharges", "TotalCharges", "tenure"]
    for col in merge_cols:
        if col in orig.columns and col in train.columns:
            tmp = orig.groupby(col)["Churn_bin"].mean().reset_index()
            tmp.columns = [col, f"orig_{col}_churn_rate"]
            train = train.merge(tmp, on=col, how="left")
            test  = test.merge(tmp, on=col, how="left")

    # Yeni: 2'li kombinasyonlar — çift kolon groupby
    combo_pairs = [
        ("tenure", "Contract"),
        ("MonthlyCharges", "InternetService"),
        ("tenure", "PaymentMethod"),
    ]
    for col1, col2 in combo_pairs:
        if col1 in orig.columns and col2 in orig.columns:
            tmp = orig.groupby([col1, col2])["Churn_bin"].mean().reset_index()
            tmp.columns = [col1, col2, f"orig_{col1}_{col2}_churn_rate"]
            # orig'daki kategorikleri de encode etmek gerekiyor
            if orig[col2].dtype == object:
                le2 = LabelEncoder()
                le2.fit(orig[col2].astype(str))
                try:
                    tmp[col2] = le2.transform(tmp[col2].astype(str))
                    train = train.merge(tmp, on=[col1, col2], how="left")
                    test  = test.merge(tmp, on=[col1, col2], how="left")
                except:
                    pass
    print(f"✅ Orijinal veri feature'ları eklendi. Yeni feature sayısı: {train.shape[1]}")

# ── DENEY 3: Rank-based normalizasyon ──
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in [TARGET, "id"]]

for col in num_cols:
    # Her sayısal feature için percentile rank ekle
    combined = pd.concat([train[col], test[col]], axis=0)
    ranks = combined.rank(pct=True)
    train[f"{col}_rank"] = ranks.iloc[:len(train)].values
    test[f"{col}_rank"]  = ranks.iloc[len(train):].values

print(f"✅ Rank feature'ları eklendi. Toplam feature: {train.shape[1]}")

# ─────────────────────────────────────────────
# MODEL KARŞILAŞTIRMA: 3 max_depth deneyi
# ─────────────────────────────────────────────
feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]
X = train[feature_cols].copy()
y = (train[TARGET] == "Yes").astype(int) if train[TARGET].dtype == object else train[TARGET].astype(int)
X_test = test[feature_cols].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

for depth in [1, 3, 4]:  # Deney 1: depth karşılaştırması
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        model = xgb.XGBClassifier(
            n_estimators=2000,
            learning_rate=0.02,
            max_depth=depth,
            subsample=0.8,
            colsample_bytree=0.7,
            min_child_weight=10,
            reg_alpha=0.5,
            reg_lambda=2.0,
            eval_metric="auc",
            use_label_encoder=False,
            random_state=42,
            tree_method="hist",
            verbosity=0,
            early_stopping_rounds=150,
        )
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            verbose=False,
        )
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        preds += model.predict_proba(X_test)[:, 1] / 5

    auc = roc_auc_score(y, oof)
    results[depth] = {"auc": auc, "preds": preds}
    print(f"  max_depth={depth} → OOF AUC: {auc:.5f}")

# ─────────────────────────────────────────────
# EN İYİ DEPTH İLE SUBMİSYON
# ─────────────────────────────────────────────
best_depth = max(results, key=lambda d: results[d]["auc"])
best_auc   = results[best_depth]["auc"]
best_preds = results[best_depth]["preds"]

print(f"\n🏆 En iyi depth: {best_depth} → OOF AUC: {best_auc:.5f}")
print(f"   (Senin önceki en iyin 0.91378 idi)")

if best_auc > 0.9138:
    print("   ✅ YENİ REKOR! Kaggle'a yükle!")
else:
    print("   ⚠️  Geçemedi — ama depth=1 sonucuna bak, farklı feature seti gerekebilir.")

submission = pd.DataFrame({
    "id": test[ID_COL],
    "Churn": best_preds,
})
submission.to_csv("submission_new.csv", index=False)
print(f"\n📁 submission_new.csv oluşturuldu (depth={best_depth})")

# Her depth için ayrı submission
for depth, res in results.items():
    pd.DataFrame({"id": test[ID_COL], "Churn": res["preds"]}).to_csv(
        f"submission_depth{depth}.csv", index=False
    )
print("📁 submission_depth1.csv, submission_depth3.csv, submission_depth4.csv oluşturuldu")
print("\n💡 İpucu: depth=1 CV yüksekse orijinal veri sinyali yakalanıyor demektir.")
print("          depth=3-4 CV yüksekse sentetik veri artefaktları da kullanılıyor.")

✅ Orijinal IBM verisi yüklendi.
✅ Orijinal veri feature'ları eklendi. Yeni feature sayısı: 32
✅ Rank feature'ları eklendi. Toplam feature: 62
  max_depth=1 → OOF AUC: 0.91214
  max_depth=3 → OOF AUC: 0.91592
  max_depth=4 → OOF AUC: 0.91648

🏆 En iyi depth: 4 → OOF AUC: 0.91648
   (Senin önceki en iyin 0.91378 idi)
   ✅ YENİ REKOR! Kaggle'a yükle!

📁 submission_new.csv oluşturuldu (depth=4)
📁 submission_depth1.csv, submission_depth3.csv, submission_depth4.csv oluşturuldu

💡 İpucu: depth=1 CV yüksekse orijinal veri sinyali yakalanıyor demektir.
          depth=3-4 CV yüksekse sentetik veri artefaktları da kullanılıyor.
